<a href="https://colab.research.google.com/github/s23970-pj/szerokosc_rzek_zpb/blob/adrian-branch/ZPB.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import ee
import pandas as pd
import requests
from google.colab import drive
import torchvision
from PIL import Image

!pip install earthengine-api geemap
import geemap

class SateliteImages(Dataset):
    """SateliteDataset"""
    def __init__(self, csv_file, root_dir, transform=None):
        self.satelite_data = pd.read_csv(csv_file)

        self.transform = transform

    def __len__(self):
        return len(self.satelite_data)

    def __getitem__(self, idx):
        if torch.is_tensor(idx):
            idx = idx.tolist()

        lon = satelite_data.iloc[idx]['longitude']
        lat = satelite_data.iloc[idx]['latitude']

        width = satelite_data.iloc[idx]['width']

        #buffer w okół punktu
        bufferSize = 1500

        # Define region (buffer around point)
        region = ee.Geometry.Point(lon, lat).buffer(bufferSize).bounds()

        # Bierzemy kolekcję z przekroju z zachmurzeniem mniej niż 20%
        collection = (
            ee.ImageCollection("COPERNICUS/S2_HARMONIZED")
            .filterBounds(region)
            .filterDate("2024-01-01", "2024-06-30")
            .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20))
        )

        composite = collection.median().clip(region)

        sample = {'image': image, 'width': width}

        if self.transform:
            sample = self.transform(sample)

        return sample

NameError: name 'Dataset' is not defined

In [ ]:

import os

drive.mount('/content/drive')
# Trigger the authentication flow.
ee.Authenticate()
ee.Initialize(project='1059694645023')

PROJECT_DIR = "tutaj sciezka do googla" #globalnie zdefiniowana sciezka
DATA_DIR = f"{PROJECT_DIR}/data"
OUTPUT_DIR = f"{PROJECT_DIR}/outputs"

os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

%cd $PROJECT_DIR

df = pd.read_csv(f"Above10m.csv") #sciezka pod csvke

imageNumber = 200

lon = df.iloc[imageNumber]['longitude']
lat = df.iloc[imageNumber]['latitude']

#buffer w okół punktu
bufferSize = 1500

# Define region (buffer around point)
region = ee.Geometry.Point(lon, lat).buffer(bufferSize).bounds()

# Bierzemy kolekcję z przekroju z zachmurzeniem mniej niż 20%
collection = (
    ee.ImageCollection("COPERNICUS/S2_HARMONIZED")
    .filterBounds(region)
    .filterDate("2024-01-01", "2024-06-30")
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20))
)

composite = collection.median().clip(region)

# Obliczanie MNDWI (Green: B3, SWIR: B11)
mndwi = composite.normalizedDifference(['B3', 'B11'])

# Jeśli chcesz dodać to do mapy w geemap
Map = geemap.Map()
Map.centerObject(region, 13)

mndwi_vis = {
    'min': -1,
    'max': 1,
    'palette': ['brown', 'white', 'blue']
}

Map.addLayer(mndwi, mndwi_vis, 'MNDWI Water Index')
Map # Wyświetlenie mapy

output_path = '/content/drive/MyDrive/earth_engine_image.tif'

bands = ['B4'] #kolorki tutaj

vis_params = {
    'min': 0,
    'max': 3000,
    'bands': bands
}

# Tu na dysk zapisuje indeks MNDWI
geemap.ee_export_image(
    mndwi,
    filename=f"{OUTPUT_DIR}/mndwi_result_{imageNumber}.tif",
    scale=10,
    region=region,
    file_per_band=False,
    crs="EPSG:2180"
    #dimensions='128x128'
)

print("Saved image") #zmiana